# Week 13: Loop Log & Eval Gates (the Three Loops)

> ZoroLogistics Support Bot MVP · run the spec → agentic → developer → external loop with a verifier closing every cycle.

# Requirements: none beyond shell tools (install per reference/skills/ guides)


## 0. The three loops, in one page

Andrew Ng frames 0-to-1 building as **three loops on different clocks** (see
`reference/knowledge-base/01-ai-engineering-discipline.md`):

| Loop | Cadence | What happens | Your artifact this week |
|---|---|---|---|
| **Agentic coding** | minutes | agent writes code, runs tests, iterates until the spec/verifier passes | Support Bot MVP + green verifier |
| **Developer feedback** | tens of minutes to hours | you review with fresh eyes, update the spec, add missed edge cases | spec v2 + new verifier checks |
| **External feedback** | hours to weeks | a real person uses it; issues update your vision → spec → agent | feedback form + filed issues |

The load-bearing piece is the **verifier**: a loop with no check just produces confident,
unchecked output (see `reference/knowledge-base/09-harnesses-tools.md`). This notebook builds a tiny
verifier for the Support Bot, logs the loops, and ends by printing the defect metric.


## 1. The Support Bot MVP

The MVP is deliberately small: a function that reads a support ticket's text, classifies
its category (tracking / damage / refund / documents / customs / billing), and returns a
canned reply. Build this here, or reuse the one you started in Week 12, the verifier in
§2 is what actually matters.


In [ ]:
import sys, pathlib
# zoro import cell (per week template)
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root
for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_c / 'zoro').is_dir():
        sys.path.insert(0, str(_c))
        break

from zoro import data  # noqa: E402


In [ ]:
# A tiny deterministic keyword classifier. It is NOT the point, it exists so the
# verifier in the next section has something concrete to pass or fail.
CATEGORY_KEYWORDS = {
    'tracking': ['where is', 'tracking', 'supposed to arrive', 'track my'],
    'damage': ['damaged', 'crushed', 'broken', 'smashed'],
    'refund': ['refund', 'hours late', 'days late', 'compensat', 'reimburse'],
    'documents': ['bill of lading', 'resend', 'lost the copy', 'shipping documents'],
    'customs': ['customs', 'holding', 'border', 'duties'],
    'billing': ['invoice', 'wrong weight', 'billing', 'charged'],
}

REPLIES = {
    'tracking': 'Your shipment {sid} is in transit; check the tracking id at portal.zorologistics.example.',
    'damage': 'We are sorry. File a damage claim with photos within 7 days (policy POL-002).',
    'refund': 'Refunds over $500 require supervisor approval (policy POL-002); review within 10 business days.',
    'documents': 'We will resend the bill of lading for {sid} to the contact on file.',
    'customs': 'Customs needs a commercial invoice + bill of lading; missing docs add 1 to 3 days (policy POL-004).',
    'billing': 'We will correct the invoice weight and reissue within 2 business days.',
    'unknown': 'I could not determine the issue; routing to a human agent.',
}

def classify_ticket(text):
    """Return the best-matching category, or 'unknown' when no keyword matches."""
    t = text.lower()
    scores = {cat: sum(t.count(kw) for kw in kws) for cat, kws in CATEGORY_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'unknown'

def answer_ticket(text):
    """Classify a ticket and return a structured answer."""
    import re
    cat = classify_ticket(text)
    m = re.search(r'S[0-9]{7}', text)
    sid = m.group(0) if m else None
    reply = REPLIES[cat].format(sid=sid or 'unknown')
    return {'category': cat, 'shipment_id': sid, 'reply': reply}

# Smoke test on one hand-written ticket:
print(answer_ticket('Where is my shipment S0000042? It was supposed to arrive 2026-03-01.'))


In [ ]:
# Eval set: 60 seeded tickets from the same generator the whole program uses.
_eval_tickets = data.support_tickets(n=60, seed=123)

# Ground-truth categories come straight from the generator.
_eval_pairs = [(row['text'], row['category']) for _, row in _eval_tickets.iterrows()]
print('eval set size:', len(_eval_pairs))
print(_eval_pairs[0])


## 2. The verifier: unit checks + a mini eval

A **verifier** is the executable definition of *done* (see
`reference/knowledge-base/01-ai-engineering-discipline.md`). We use two layers, the way a real coding
loop would:

1. **Unit checks**: a handful of hand-written assertions with known answers (pytest-style,
   but run by a 10-line runner so the notebook needs no pytest dependency).
2. **A mini eval**: classification accuracy over 60 seeded tickets from `zoro.data`.

The bot is not done until *both* layers pass; a defect is a failed check or a wrong
prediction.


In [ ]:
def run_checks(checks):
    """Run a list of (name, callable) checks; return (passed, failed)."""
    passed, failed = [], []
    for name, fn in checks:
        try:
            fn()
            passed.append(name)
        except AssertionError as e:
            failed.append((name, str(e)))
    return passed, failed

# --- Unit checks (pytest-style, no pytest needed) --------------------------
def _check_tracking():
    assert classify_ticket('Where is my shipment S0000001? It was supposed to arrive 2026-03-01.') == 'tracking'

def _check_damage():
    assert classify_ticket('The pallet for S0000002 arrived damaged. The electronics boxes are crushed.') == 'damage'

def _check_refund():
    assert classify_ticket('I want a refund for shipment S0000003. It arrived 48 hours late.') == 'refund'

def _check_customs():
    assert classify_ticket('Customs is holding S0000004 at Long Beach. What documents do you need?') == 'customs'

def _check_unknown():
    assert classify_ticket('hello') == 'unknown'

def _check_answer_shape():
    a = answer_ticket('I want a refund for shipment S0000003. It arrived 48 hours late.')
    assert set(a.keys()) == {'category', 'shipment_id', 'reply'}
    assert a['category'] == 'refund'

UNIT_CHECKS = [
    ('tracking lookup', _check_tracking),
    ('damage lookup', _check_damage),
    ('refund lookup', _check_refund),
    ('customs lookup', _check_customs),
    ('unknown input', _check_unknown),
    ('answer shape', _check_answer_shape),
]

passed, failed = run_checks(UNIT_CHECKS)
print('unit checks:', len(passed), '/', len(UNIT_CHECKS), 'passed')
for name, err in failed:
    print('  FAIL', name, ':', err)

# --- Mini eval (classification accuracy) -----------------------------------
correct = sum(1 for text, true_cat in _eval_pairs if classify_ticket(text) == true_cat)
n_eval = len(_eval_pairs)
print('mini eval:', correct, '/', n_eval, 'correct =', round(correct / n_eval, 3))


## 3. The agentic loop (headless)

A coding agent can run **headless**, `claude -p '<task>'`, which is how you script the
agentic loop into CI or a pipeline (see `reference/skills/claude-code.md`). The cell below runs the
pattern with a graceful skip: if `claude` is missing it prints the manual fallback instead
of failing, so the notebook still runs on any machine.


In [ ]:
%%bash
echo '== headless agentic-loop example (claude -p) =='
if command -v claude >/dev/null 2>&1; then
  claude -p 'Read zoro/data.py and this notebook. Add one more unit check to the verifier for the billing category, then report whether all checks pass.' 2>&1 | head -n 40
else
  echo 'claude not installed, see skills/claude-code.md.'
  echo 'Manual fallback: paste the task above into your harness, run the build, and record the result in the loop log below.'
fi


## 4. The loop log

A loop you cannot measure is a vibe. The **loop log** is one row per cycle, recording the
loop (agentic / developer / external), timestamps, tokens, and whether a defect surfaced.
This is the evidence the Friday deliverable quotes, the same *configuration record* habit
from the systems-engineering spine (`reference/knowledge-base/01-ai-engineering-discipline.md`).


In [ ]:
import pandas as pd
from datetime import datetime, timezone

# The loop log: append one dict per cycle. You fill the real numbers as you work.
loop_log = []

def log_loop(loop, event, tokens_in=0, tokens_out=0, defect=None, notes=''):
    loop_log.append({
        'ts_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'loop': loop,               # 'agentic' | 'developer' | 'external'
        'event': event,
        'tokens_in': tokens_in,
        'tokens_out': tokens_out,
        'defect': defect,           # e.g. 'wrong category on customs text', or None
        'notes': notes,
    })
    return loop_log

# Seeded example rows (replace with your real cycles):
log_loop('agentic', 'initial build of classify_ticket', 0, 0, 'refund text mis-flagged as tracking', 'example')
log_loop('agentic', 'fix keyword list; verifier re-run', 0, 0, None, 'example')
log_loop('developer', 'fresh-eyes review; added unknown-input check', 0, 0, 'no check for unknown text', 'example')

loop_frame = pd.DataFrame(loop_log)
print('cycles logged:', len(loop_frame), '| defects surfaced:', int(loop_frame['defect'].notna().sum()))
loop_frame


## 5. Developer loop: review prompts

The developer loop is your **context advantage**: you know more than the agent about the
users and the operating context. Run these prompts with fresh eyes, *before* asking the
agent to change anything, then write the resulting decisions into the spec, not into a
chat message that will be lost.


**Developer-loop review prompts (answer in writing, not in your head):**

1. **What did the agent assume that the spec never said?** Name it, then decide: add it to
   the spec, or mark it out of scope.
2. **Which input did nobody test?** e.g. empty string, a ticket with two shipment ids, a
   non-ASCII customer name, an `S` id in the middle of prose.
3. **What is the blast radius of a wrong answer?** A misclassified *damage* ticket delays a
   real claim; a wrong *refund* spends money. Does the verifier cover that?
4. **What would a human agent do that the bot cannot?** Write that as a *route to human*
   rule and a test.
5. **Is the verifier measuring what the spec promises?** If the spec says *classify
   correctly*, a check that only asserts the function *returns a string* is theatre.


## 6. External feedback: form template

The external loop is the only one with **reality** as its verifier. Hand the bot (or a
printed transcript) to someone real and collect structured feedback, then convert every
issue into either a spec change or a new eval case.


**External-feedback form (copy into a doc and send):**

- Name / role: ________________
- Task you tried (in your own words): ________________
- Did it work the first time? yes / no
- If no, what specifically went wrong? (paste the exact input) ________________
- What did you expect it to say? ________________
- Severity (1 = cosmetic, 5 = blocked): ____
- Would you trust this for a real shipment? yes / no / only with a human check

**Triage rule after each response:** 1) file it as an issue, 2) decide spec-vs-eval,
3) add the failing input to the eval set so it can never silently regress.


## 7. Defect metric

The final cell prints the number this notebook exists for: the **verifier pass rate**
across unit checks + eval cases, plus the count of defects the verifier caught. A passing
MVP is not *looks right*, it is this number, with the sample that produced it.


In [ ]:
# Final metric: verifier pass rate = (unit checks passed + eval predictions correct)
#               / (unit checks + eval cases). Also report defects caught.
unit_total = len(UNIT_CHECKS)
unit_passed = len(passed)
eval_total = len(_eval_pairs)
eval_correct = correct

verifier_pass_rate = (unit_passed + eval_correct) / (unit_total + eval_total)
defects_caught = (unit_total - unit_passed) + (eval_total - eval_correct)

print('unit checks   :', unit_passed, '/', unit_total)
print('eval correct  :', eval_correct, '/', eval_total)
print('defects caught:', defects_caught)
print(round(verifier_pass_rate, 3))
